# ReefScan-ID: Stage 1 baseline (E1)

A small U-Net trained from scratch, compared with the pretrained SegFormer-B0 of notebook 02 on the Coralscapes validation set. The data, class mapping, crops, loss and scoring are the same as in notebook 02. The test split is not used. Run in Google Colab with a GPU runtime.

### Setup: Colab runtime and Google Drive

In [2]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))
print("CPU count: {}".format(os.cpu_count()))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))

Mounted at /content/drive
GPU: Tesla T4
CPU count: 2
Drive folder found: True


### Setup: libraries

In [3]:
!pip install -q datasets
import random
import time
import numpy as np
from PIL import Image

import datasets
print("torch version: {}".format(torch.__version__))
print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))

torch version: 2.14.1+cu130
datasets version: 4.8.5
numpy version: 2.4.6


### Settings

The learning rate is for a model trained from scratch. Notebook 02 used 6e-5 because its encoder was pretrained. The weight decay, the crop size, the batch size and the validation size are the same as in notebook 02.

In [4]:
crop_size = 512
batch_size = 8
num_workers = 2
val_height = 512
val_width = 1024
val_batch_size = 4
seed = 0

base_width = 32
levels = 4
learning_rate = 1e-3
weight_decay = 0.01
timing_steps = 30

# Reference values printed by notebook 02
segformer_params = 3714915
segformer_pixels = 3181379584
image_mean_values = [0.485, 0.456, 0.406]
image_std_values = [0.229, 0.224, 0.225]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("Crops: {} by {}, batch size {}, workers {}".format(crop_size, crop_size, batch_size, num_workers))
print("Validation photos resized to {} by {}, batch size {}".format(val_height, val_width, val_batch_size))
print("U-Net: {} levels, base width {}".format(levels, base_width))
print("Optimiser for the baseline: AdamW, learning rate {}, weight decay {}, linear decay to 0".format(
    learning_rate, weight_decay))
print("Notebook 02 used: AdamW, learning rate 6e-05, weight decay 0.01, linear decay to 0")

Crops: 512 by 512, batch size 8, workers 2
Validation photos resized to 512 by 1024, batch size 4
U-Net: 4 levels, base width 32
Optimiser for the baseline: AdamW, learning rate 0.001, weight decay 0.01, linear decay to 0
Notebook 02 used: AdamW, learning rate 6e-05, weight decay 0.01, linear decay to 0


## Section 1: Data and model

Question: does the baseline see the same data, class mapping, crops, loss and scoring as the SegFormer-B0 run of notebook 02, how large is the U-Net next to SegFormer-B0, and how long does one epoch take on this GPU?

Checks: load Coralscapes from the Drive cache, rebuild the class mapping and the class weights, build the crop and validation loaders, build the U-Net and print its parameter count, run one forward pass, score the untrained model once, then time 30 training steps and one validation pass to estimate the seconds per epoch. Only the train and validation splits are used.

### Load the dataset

In [5]:
raw_dir = base_dir + "/RawData"

from datasets import load_dataset

print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

Cache folder exists: True


README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})
Loading took 45.6 seconds


### Class names from the dataset card

Ids and names copied from notebook 02, which copied them from the table on the Coralscapes dataset card.

In [6]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Build the mapping array

In [7]:
coral_ids = [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]
substrate_ids = [5, 10, 12, 18]
group_names = {0: "coral", 1: "substrate", 2: "other"}
ignore_id = 255

# Values that are not on the card stay at the ignore id
mapping = np.full(256, ignore_id, dtype=np.uint8)
for value in class_names:
    mapping[value] = 2
mapping[coral_ids] = 0
mapping[substrate_ids] = 1

for group_id, group_name in group_names.items():
    names = [class_names[v] for v in class_names if mapping[v] == group_id]
    print("{} ({} ids): {}".format(group_name, len(names), names))
print("Mask value 0 maps to: {}".format(mapping[0]))

coral (19 ids): ['other coral dead', 'other coral bleached', 'other coral alive', 'massive/meandering bleached', 'massive/meandering alive', 'branching bleached', 'branching dead', 'millepora', 'branching alive', 'massive/meandering dead', 'acropora alive', 'turbinaria', 'table acropora alive', 'pocillopora alive', 'table acropora dead', 'meandering bleached', 'stylophora alive', 'meandering alive', 'meandering dead']
substrate (4 ids): ['sand', 'algae covered substrate', 'unknown hard substrate', 'rubble']
other (16 ids): ['seagrass', 'trash', 'human', 'transect tools', 'fish', 'other animal', 'background', 'dark', 'transect line', 'clam', 'sea cucumber', 'sponge', 'anemone', 'sea urchin', 'crown of thorn', 'dead clam']
Mask value 0 maps to: 255


### Count train pixels per mask value

In [8]:
pixel_counts = {}

start = time.time()
counts = np.zeros(256, dtype=np.int64)
masks = ds["train"].select_columns(["label"])
for row in masks:
    mask = np.array(row["label"])
    counts += np.bincount(mask.ravel(), minlength=256)
pixel_counts["train"] = counts

print("train: {} masks counted, {} pixels, {:.0f} seconds".format(masks.num_rows, counts.sum(), time.time() - start))
print("Same pixel total as notebook 02: {}".format(int(counts.sum()) == segformer_pixels))

train: 1517 masks counted, 3181379584 pixels, 62 seconds
Same pixel total as notebook 02: True


### Class weights and loss

In [9]:
train_counts = pixel_counts["train"]
group_pixels = np.array([train_counts[mapping == group_id].sum() for group_id in group_names])
group_share = group_pixels / group_pixels.sum()
class_weights = 1 / group_share
class_weights = class_weights / class_weights.mean()

for group_id, group_name in group_names.items():
    print("{}: {:.2f}% of labelled train pixels, weight {:.3f}".format(
        group_name, 100 * group_share[group_id], class_weights[group_id]))
print("Same weights as notebook 02 (1.076, 1.000, 0.924): {}".format(
    bool(np.allclose(class_weights.round(3), [1.076, 1.0, 0.924]))))

weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)


def segmentation_loss(logits, masks):
    upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
    return torch.nn.functional.cross_entropy(upsampled, masks, weight=weight_tensor, ignore_index=ignore_id)

coral: 30.87% of labelled train pixels, weight 1.076
substrate: 33.19% of labelled train pixels, weight 1.000
other: 35.94% of labelled train pixels, weight 0.924
Same weights as notebook 02 (1.076, 1.000, 0.924): True


### Crop dataset class

In [10]:
import random
from torch.utils.data import Dataset

crop_size = 512


class CropDataset(Dataset):
    def __init__(self, split, crop_size, flip):
        self.split = split
        self.crop_size = crop_size
        self.flip = flip

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = np.array(row["image"])
        mask = mapping[np.array(row["label"])]

        # Same random window for the image and the mask
        top = random.randint(0, image.shape[0] - self.crop_size)
        left = random.randint(0, image.shape[1] - self.crop_size)
        image = image[top:top + self.crop_size, left:left + self.crop_size]
        mask = mask[top:top + self.crop_size, left:left + self.crop_size]

        if self.flip and random.random() < 0.5:
            image = image[:, ::-1]
            mask = mask[:, ::-1]

        image = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mask.copy()).long()
        return image, mask


print("Crop size: {}".format(crop_size))

Crop size: 512


### Validation dataset class

In [11]:
class ResizeDataset(Dataset):
    def __init__(self, split, height, width):
        self.split = split
        self.height = height
        self.width = width

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = row["image"].resize((self.width, self.height), Image.BILINEAR)
        label = row["label"].resize((self.width, self.height), Image.NEAREST)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mapping[np.array(label)]).long()
        return image, mask


print("Validation dataset class defined")

Validation dataset class defined


### Data loaders

In [12]:
from torch.utils.data import DataLoader

train_data = CropDataset(ds["train"], crop_size, flip=True)
eval_data = ResizeDataset(ds["validation"], val_height, val_width)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
eval_loader = DataLoader(eval_data, batch_size=val_batch_size, shuffle=False, num_workers=num_workers)

print("Train crops per epoch: {}, batches: {}".format(len(train_data), len(train_loader)))
print("Validation photos: {}, batches: {}".format(len(eval_data), len(eval_loader)))

for name, loader in [("train", train_loader), ("validation", eval_loader)]:
    images, masks = next(iter(loader))
    print("{} batch: images {}, dtype {}, min {:.3f}, max {:.3f}".format(
        name, tuple(images.shape), images.dtype, images.min().item(), images.max().item()))
    print("{} batch: masks {}, dtype {}, values {}".format(
        name, tuple(masks.shape), masks.dtype, torch.unique(masks).tolist()))

Train crops per epoch: 1517, batches: 190
Validation photos: 166, batches: 42
train batch: images (8, 3, 512, 512), dtype torch.float32, min 0.000, max 1.000
train batch: masks (8, 512, 512), dtype torch.int64, values [0, 1, 2, 255]
validation batch: images (4, 3, 512, 1024), dtype torch.float32, min 0.000, max 1.000
validation batch: masks (4, 512, 1024), dtype torch.int64, values [0, 1, 2, 255]


### U-Net

Each level has two 3 by 3 convolutions with batch normalisation and ReLU. The encoder halves the size with max pooling between levels, the decoder doubles it with a transposed convolution and joins the skip connection of the same level. The widths double from level to level, starting at the base width.

In [13]:
from torch import nn


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels, num_classes, base_width, levels):
        super().__init__()
        widths = [base_width * 2 ** level for level in range(levels)]
        self.encoders = nn.ModuleList([DoubleConv(in_channels, widths[0])])
        for level in range(1, levels):
            self.encoders.append(DoubleConv(widths[level - 1], widths[level]))
        self.pool = nn.MaxPool2d(2)

        self.ups = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for level in range(levels - 1, 0, -1):
            self.ups.append(nn.ConvTranspose2d(widths[level], widths[level - 1], 2, stride=2))
            self.decoders.append(DoubleConv(widths[level - 1] * 2, widths[level - 1]))
        self.head = nn.Conv2d(widths[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for level, encoder in enumerate(self.encoders):
            x = encoder(x)
            if level < len(self.encoders) - 1:
                skips.append(x)
                x = self.pool(x)
        for up, decoder, skip in zip(self.ups, self.decoders, reversed(skips)):
            x = up(x)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.head(x)


model = UNet(3, len(group_names), base_width, levels).to(device)

image_mean = torch.tensor(image_mean_values, device=device).view(1, 3, 1, 1)
image_std = torch.tensor(image_std_values, device=device).view(1, 3, 1, 1)


def normalise(images):
    return (images - image_mean) / image_std


total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Level widths: {}".format([base_width * 2 ** level for level in range(levels)]))
print("U-Net parameters: {} total, {} trainable".format(total_params, trainable_params))
print("SegFormer-B0 parameters printed by notebook 02: {}".format(segformer_params))
print("U-Net parameters divided by SegFormer-B0 parameters: {:.2f}".format(total_params / segformer_params))

Level widths: [32, 64, 128, 256]
U-Net parameters: 1927075 total, 1927075 trainable
SegFormer-B0 parameters printed by notebook 02: 3714915
U-Net parameters divided by SegFormer-B0 parameters: 0.52


### One forward pass

In [14]:
images, masks = next(iter(train_loader))

model.eval()
with torch.no_grad():
    logits = model(normalise(images.to(device)))

print("Input shape: {}".format(tuple(images.shape)))
print("Logits shape: {}".format(tuple(logits.shape)))
print("Logits have the size of the masks: {}".format(tuple(logits.shape[-2:]) == tuple(masks.shape[-2:])))

Input shape: (8, 3, 512, 512)
Logits shape: (8, 3, 512, 512)
Logits have the size of the masks: True


### Validation metrics

In [15]:
group_count = len(group_names)


def evaluate(model, loader):
    confusion = torch.zeros(group_count, group_count, dtype=torch.int64, device=device)
    model.eval()
    with torch.no_grad():
        for images, masks in loader:
            masks = masks.to(device)
            logits = model(normalise(images.to(device)))
            upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled.argmax(dim=1)

            # Pixels with the ignore id are left out
            keep = masks != ignore_id
            pairs = masks[keep] * group_count + predicted[keep]
            confusion += torch.bincount(pairs, minlength=group_count ** 2).reshape(group_count, group_count)

    intersection = confusion.diag().float()
    union = confusion.sum(dim=0).float() + confusion.sum(dim=1).float() - intersection
    return (intersection / union).cpu().numpy()


start = time.time()
iou = evaluate(model, eval_loader)
print("Model before training")
for group_id, group_name in group_names.items():
    print("{} IoU: {:.4f}".format(group_name, iou[group_id]))
print("mIoU: {:.4f}".format(iou.mean()))
print("One validation pass took {:.1f} seconds".format(time.time() - start))

Model before training
coral IoU: 0.2276
substrate IoU: 0.0000
other IoU: 0.0000
mIoU: 0.0759
One validation pass took 29.2 seconds


### Step timing and estimated seconds per epoch

The training steps run on a copy of the model, so the model above stays untouched. The data loading time is the time the training loop waits for the next batch, so it already includes the overlap with the model step.

In [16]:
import copy

timing_model = copy.deepcopy(model)
timing_model.train()
timing_optimizer = torch.optim.AdamW(timing_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

data_seconds = []
step_seconds = []
batches = iter(train_loader)
for _ in range(timing_steps):
    start = time.time()
    images, masks = next(batches)
    data_seconds.append(time.time() - start)

    start = time.time()
    loss = segmentation_loss(timing_model(normalise(images.to(device))), masks.to(device))
    timing_optimizer.zero_grad()
    loss.backward()
    timing_optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    step_seconds.append(time.time() - start)

start = time.time()
evaluate(timing_model, eval_loader)
validation_seconds = time.time() - start

first_step = data_seconds[0] + step_seconds[0]
steady_step = np.mean(data_seconds[1:]) + np.mean(step_seconds[1:])
steps_per_epoch = len(train_loader)
epoch_seconds = first_step + (steps_per_epoch - 1) * steady_step + validation_seconds

print("Steps timed: {}, batch size: {}".format(timing_steps, batch_size))
print("Step 1, includes start-up: data loading {:.2f} seconds, model step {:.2f} seconds".format(
    data_seconds[0], step_seconds[0]))
print("Steps 2 to {}: data loading {:.2f} seconds per step".format(timing_steps, np.mean(data_seconds[1:])))
print("Steps 2 to {}: model step {:.2f} seconds per step".format(timing_steps, np.mean(step_seconds[1:])))
print("Steps 2 to {}: total {:.2f} seconds per training step".format(timing_steps, steady_step))
print("One validation pass: {:.1f} seconds".format(validation_seconds))
if device == "cuda":
    print("Peak GPU memory during the training steps: {:.1f} GB".format(torch.cuda.max_memory_allocated() / 1024 ** 3))
print("Estimated seconds per epoch: {:.0f} ({} steps and one validation pass)".format(epoch_seconds, steps_per_epoch))
print("Estimated training time for a number of epochs on {}:".format(gpu_name))
for epochs_option in [10, 20, 30, 40, 50]:
    print("  {} epochs: about {:.0f} minutes".format(epochs_option, epochs_option * epoch_seconds / 60))

Steps timed: 30, batch size: 8
Step 1, includes start-up: data loading 2.71 seconds, model step 1.25 seconds
Steps 2 to 30: data loading 0.46 seconds per step
Steps 2 to 30: model step 0.64 seconds per step
Steps 2 to 30: total 1.10 seconds per training step
One validation pass: 21.4 seconds
Peak GPU memory during the training steps: 6.3 GB
Estimated seconds per epoch: 233 (190 steps and one validation pass)
Estimated training time for a number of epochs on Tesla T4:
  10 epochs: about 39 minutes
  20 epochs: about 78 minutes
  30 epochs: about 117 minutes
  40 epochs: about 156 minutes
  50 epochs: about 194 minutes


**Observation**: From the result, we can see that the U-Net has 1,927,075 parameters, around half of SegFormer-B0. Before training, it reached 0.0759 mIoU on Coralscapes validation, so this is not the final baseline performance. On the Tesla T4, one epoch is estimated to take 233 seconds, and we still need to train the model before comparing both architectures.